# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [ ]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [ ]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# Lists to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None,  # Default to None (indicating missing content)
        "error_code": None  # New field for failed downloads
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        file_entry["error_code"] = "Invalid URL"
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        file_entry["error_code"] = response.status_code  # Store specific HTTP error code
        print(f"❌ Failed to fetch {direct_url} - HTTP {response.status_code}: {http_err}")
        failed_downloads.append(file_entry)
    except requests.exceptions.RequestException as e:
        file_entry["error_code"] = "Request Error"
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")

# Read the Training Data

In [5]:
import pandas as pd
import numpy as np

In [31]:
#Converted to long format in R
df_train_raw = pd.read_csv("../data/annotations_long.csv")
keep = ["row_id", "file_hash","type","subtype_confidence","notes_free_text","label"]
df_train = df_train_raw[keep].copy()

In [32]:
df_train

,row_id,file_hash,type,subtype_confidence,notes_free_text,label
0,1,f8be35d0c20d1b1f3de4c44323e1780ee24f06893b6364...,I Na,3 - Highly confident,NaN,i_na_persistent
1,2,e97ca8a7f9734805832e5ae75442d19d3d7796b9a24190...,I K,2 - Mildly confident,STATES n l taul = 0.26*(v+50)/qtl,i_k_a_type_slow
2,4,606423f8f4a4f406f3387c7ee6f142bd121237272c347d...,Receptor,3 - Highly confident,NaN,r_gaba_type_a
3,5,99713c0032634e96cc7cde2dce02d8aa2baf75ad4cc835...,Receptor,3 - Highly confident,NaN,r_gaba_general
4,6,2d34ae241d628e7f25374c1bd84a8138cedc6a54689568...,Receptor,3 - Highly confident,mix,r_glutamate_ampa
...,...,...,...,...,...,...
414,462,ed87e94223dec25e43020f389207c0e362361efd05f3b0...,I Na,3 - Highly confident,NaN,i_na_transient
415,463,414840eea6fe6455d3377f99f43543155d2e4771b8cf8c...,I Ca,3 - Highly confident,NaN,i_ca_t_type_lt
416,464,43cdc726cbc3bc20eb08263de0028f8bfba35edba720eb...,Exclude,3 - Highly confident,NaN,exclude_accumulation_mechanism
417,467,abd4902abc5ad78269b6b9a4cc6d430bdd2bdd777cafd2...,I K,3 - Highly confident,NaN,i_k_delayed_rectifier


# Feature Engineering

In [33]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
pd.set_option("display.max_columns", None)

In [34]:
def View(df, rows=None, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [35]:
# Function to extract mod directory from the URL
def get_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def get_fname(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def get_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to extract all TITLE occurrences from .mod content
def get_title(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"^TITLE\s+([^\n:]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None

# Function to extract all COMMENT sections from .mod content
def get_comment(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"COMMENT\s+(.*?)(?:\s+ENDCOMMENT|\Z)", content, re.DOTALL)  
    return matches if matches else None

# Function to extract all SUFFIX occurrences from .mod content
def get_suffix(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"SUFFIX\s+([^\n:\s]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None


def get_use_ion(content):
    """
    Extracts the ion names used in the 'USEION' statements from NEURON mod file content.

    Parameters:
    - content (str): The content of the .mod file.

    Returns:
    - list: A list of ions used in 'USEION' statements, or None if none are found.
    """
    if pd.isna(content):  
        return None
    
    # Find all occurrences of USEION followed by an ion name
    matches = re.findall(r"USEION\s+(\w+)", content, re.MULTILINE)

    return matches if matches else None


# Function to extract all ions listed after READ but stopping before WRITE, USEION, RANGE, GLOBAL, NONSPECIFIC_CURRENT, or VALENCE
def get_read_ion(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?=\s+(?:WRITE|USEION|RANGE|GLOBAL|NONSPECIFIC_CURRENT|VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return read_ions if read_ions else None  


# Function to extract all ions listed after WRITE, stopping before VALENCE
def get_write_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"WRITE\s+([^\n:]+?)(?=\s+(?:VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    write_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return write_ions if write_ions else None  


def write_current_yn(ions):
    """
    Checks if mod_write_ion contains an ion that starts with 'i' (indicating a current).

    Args:
        write_ions (list): List of ions written in the mod file.

    Returns:
        int: 1 if any ion starts with 'i', otherwise 0.
    """
    if not ions:  # Handle empty lists or None
        return 0

    return int(any(ion.startswith("i") for ion in ions))


# Function to extract all NONSPECIFIC currents
def get_nonspecific_current(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"NONSPECIFIC_CURRENT\s+([^\n:]*)", content)

    if not matches:
        return None

    nonspecific_currents = [curr.strip() for match in matches for curr in re.split(r"[,\s]+", match) if curr]

    return nonspecific_currents if nonspecific_currents else None  

#todo: should we assume we only want active variables or also extract ones that are commented out?
# Function to extract RANGE variables based on mode
def get_range(content, mode="active"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")


# Function to extract only active RANGE variables, stopping at colons and the end of the line
def get_range(content):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract all RANGE statements (each line separately), stopping before colons
    matches = re.findall(r"^\s*RANGE\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    # Process active RANGE variables, ensuring they don't capture anything past the colon
    active_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return active_vars if active_vars else None  # Return only active variables
    
# Function to extract parameter names and values as a dictionary
def get_parameter(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"PARAMETER\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    param_dict = {}
    
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore commented-out lines
                continue
            param_match = re.match(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line)
            if param_match:
                param_name, param_value = param_match.groups()
                param_dict[param_name] = float(param_value)  

    return param_dict if param_dict else None  

# Function to extract only active STATE variables, ignoring comments (`:`) and unit values `(mV)`, etc.
def get_state(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"STATE\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    state_vars = []
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore fully commented-out lines
                continue
            line = re.split(r"\s*:\s*", line)[0]  # Remove inline comments (anything after `:`)
            clean_line = re.sub(r"\([^)]*\)", "", line).strip()  # Remove unit values
            if clean_line:
                state_vars.append(clean_line)

    return state_vars if state_vars else None  


# Function to extract only active GLOBAL variables, ignoring commented-out (`:`) ones
def get_global(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"^\s*GLOBAL\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    global_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return global_vars if global_vars else None  


def get_net_receive(content):
    """
    Extracts all NET_RECEIVE block arguments from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        list or None: A list of extracted NET_RECEIVE arguments, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Find all occurrences of NET_RECEIVE and extract arguments
    matches = re.findall(r"^\s*NET_RECEIVE\s*\(\s*([\w, ]+)\s*\)", content, re.MULTILINE)

    if not matches:
        return None

    net_receive_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return net_receive_vars if net_receive_vars else None


def get_include(content):
    """
    Extracts the filename in the INCLUDE statement from MOD file content, ignoring comments.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        list or None: A list of extracted INCLUDE filenames, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Find all occurrences of INCLUDE, ensuring commented-out ones (with ':') are ignored
    matches = re.findall(r"^\s*INCLUDE\s+\"([^\"]+)\"", content, re.MULTILINE)

    return matches if matches else None


def get_point_process(content):
    """
    Extracts the POINT_PROCESS name from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        str or None: The extracted POINT_PROCESS name, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Extract the POINT_PROCESS name, ignoring comments
    match = re.search(r"^\s*POINT_PROCESS\s+([^\n:]+)", content, re.MULTILINE)

    return match.group(1).strip() if match else None


    
# Function to extract webpage heading
def get_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def get_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def get_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def get_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found




def has_x86(url):
    """
    Checks if the given URL contains 'x86_64'.

    Parameters:
    - url (str): The URL to check.

    Returns:
    - int: 1 if 'x86_64' is present in the URL, 0 otherwise.
    """
    if pd.isna(url):  
        return None  # Return None for missing URLs
    return 1 if "x86_64" in url else 0


def has_error(error_code):
    """
    Checks if the given error code is not NaN (i.e., an error is present).

    Parameters:
    - error_code: The error code to check.

    Returns:
    - int: 1 if an error code is present (not NaN), 0 otherwise.
    """
    return 1 if error_code is not None else 0

def has_electrode_or_clamp(mod_name, content):
    """
    Checks whether 'clamp' is present in the mod file name OR 
    'ELECTRODE_CURRENT' is present in the NEURON mod file content.

    Parameters:
    - mod_name (str): The name of the .mod file.
    - content (str): The content of the .mod file.

    Returns:
    - int: 1 if either 'clamp' is in the mod name OR 'ELECTRODE_CURRENT' is in the content, 0 otherwise.
    """
    if pd.isna(mod_name) and pd.isna(content):
        return None  # Return None if both are missing

    has_clamp = bool(re.search(r"clamp", str(mod_name), re.IGNORECASE)) if pd.notna(mod_name) else False
    has_electrode = bool(re.search(r"\bELECTRODE_CURRENT\b", str(content))) if pd.notna(content) else False

    return 1 if has_clamp or has_electrode else 0

def map_ion(value):
    value = value.lower()  # Normalize to lowercase

    # Define regex-based categorization rules
    patterns = [
        (r'ca.*i$', 'ca_i'),
        (r'ca.*o$', 'ca_o'),
        (r'cl.*i$', 'cl_i'),
        (r'cl.*o$', 'cl_o'),
        (r'k.*i$', 'k_i'),
        (r'k.*o$', 'k_o'),
        (r'na.*i$', 'na_i'),
        (r'na.*o$', 'na_o'),
        (r'hco3.*i$', 'other_i'),
        (r'hco3.*o$', 'other_o'),
        (r'mgi$', 'mg_i'),  
        (r'mgo$', 'mg_o'),  
        (r'^img$', 'i_mg'),  
        (r'^emg$', 'e_mg'),
        (r'^e.*ca', 'e_ca'),
        (r'^e.*k', 'e_k'),
        (r'^e.*na', 'e_na'),
        (r'^e.*mg', 'e_mg'),
        (r'^e.*', 'e_other'),
        (r'^i.*ca', 'i_cal'),
        (r'^i.*k', 'i_k'),
        (r'^i.*cl', 'i_cl'),
        (r'^i.*na$', 'i_na'),  # FIX: Ensure "ina" is classified as "i_na"
        (r'^i.*mg', 'i_mg'),
        (r'^i.*', 'i_other'),
        (r'.*i$', 'other_i'),  # General rule: Anything ending in "i" is "other_i"
        (r'.*o$', 'other_o')   # General rule: Anything ending in "o" is "other_o"
    ]
    # Apply the regex patterns
    for pattern, category in patterns:
        if re.search(pattern, value):
            return category

    return "unknown"  # Default category if no match is found

def count_states(df, column_name="state"):
    """Counts the number of states in each row of the specified column."""
    df["count_states"] = df[column_name].apply(lambda x: len(x) if isinstance(x, list) else 0)
    return df

def map_new_label(value):
    if value in [
        'exclude_utility', 'exclude_non_neural', 'exclude_clamp',
        'exclude_accumulation_mechanism', 'exclude_artificial_cell',
        'exclude_gap_junction', 'exclude_localized_reactions'
    ]:
        return 'neither' 
    
    elif value in [
        'i_k_delayed_rectifier', 'r_glutamate_ampa', 'i_na_transient',
        'i_ca_l_type_ht', 'r_glutamate_nmda', 'i_other_mixed',
        'i_ca_t_type_lt', 'i_h_hyperpolarization_activated',
        'i_k_a_type_transient', 'i_na_persistent',
        'i_k_m_type', 'i_na_general', 'r_gaba_type_a',
        'i_k_a_type_slow'
    ]:
        return value
    elif value in ['i_cl_leak']:
        return 'i_cl'
    elif value in [
        'i_ca_ca_pump', 'i_ca_calcium_activiated_non_specific',
        'i_ca_general', 'i_ca_p_q_type', 'i_ca_q_type', 'i_ca_r_type'
    ]:
        return 'i_ca'
    elif value in [
        'i_k_ca_activated_general_bk_ik_sk_and_i_ahp_currents', 'i_k_ca_activated_bk',
        'i_k_ahp', 'i_k_ca_activated_sk'
    ]:
        return 'i_k_ca'
    
    elif value in [
        'i_k_c_type', 'i_k_general', 'i_k_ht', 'i_k_ir',
        'i_k_leak', 'i_k_lt', 'i_k_resistent_persistent', 'i_kx_photoreceptor'
    ]:
        return 'i_k'
    
    elif value in [
        'i_na_leak', 'i_na_slow_inactivation', 'i_na_general'
    ]:
        return 'i_na'
    
    elif value in [
        'i_other_channelrhodopsin', 'i_other_cng', 'i_other_kcc2_transporter',
        'i_other_modulator_activated', 'i_other_na_ca_exchanger',
        'i_other_na_k_pump','i_nonspecific','i_leak_general'
    ]:
        return 'i_other_mixed'
    
    elif value in ['r_gaba_type_a', 'r_gaba_general', 'r_gaba_type_b']:
        return 'r_gaba'
    
    elif value in ['r_acetylcholine_general', 'r_amino_acid_glutamate']:
        return 'r_other'
    
    return 'neither'

def map_broad_label(value):
    if value == "I K":
        return "i_k"
    elif value == "Exclude":
        return "neither"
    elif value == "Receptor":
        return "receptor"
    elif value == "I Na":
        return "i_na"
    elif value == "I Ca":
        return "i_ca"
    elif value in ["I Other/Mixed","I Cl","I H"]:
        return "i_other"
    else:
        return "check"  # Default label for unknown values

def get_tau(param_dict):
    if not isinstance(param_dict, dict):
        return None, None  # Return separate None values for direct unpacking

    # Extract values where the key contains 'tau'
    tau_values = [v for k, v in param_dict.items() if 'tau' in k.lower()]
    
    # If no tau values found, return (None, None)
    if not tau_values:
        return None, None
    
    # Compute min and max
    return min(tau_values), max(tau_values)


def get_e(param_dict):
    if not isinstance(param_dict, dict):
        return [None, None]  # Handle cases where the value is not a dictionary

    # Regex pattern to match variations of reversal potential names
    pattern = re.compile(r"^(e|rev|v|shift).*", re.IGNORECASE)

    # Extract values where the key matches the pattern
    e_values = [v for k, v in param_dict.items() if pattern.match(k)]

    # If no values found, return [None, None]
    if not e_values:
        return [None, None]

    # Compute min and max reversal potential
    return min(e_values), max(e_values)

In [36]:
# Load JSON into DataFrame
df_all = pd.read_json("../data/mod_files.json")
#df_all.set_index("row_id", inplace=True)
# Add exclusion criteria as flags
df_all["exclude_error"] = df_all["error_code"].apply(has_error)
df_all["exclude_x86"] = df_all["url"].apply(has_x86)
df_all['dataset'] = df_all['file_hash'].isin(df_train['file_hash']).map({True: 'train', False: 'test'})

In [37]:
df_train2 = df_train.merge(df_all, on=["row_id","file_hash"], how="left")

In [ ]:
#df_train2.loc[df_train2["exclude_x86"]==1].index == df_train2.loc[df_train2["label"]=="exclude_old_architecture"].index

In [38]:
df_train2["file_hash"].nunique()

407

In [39]:
df_train3 = df_train2.loc[(df_train2["exclude_error"] != 1) & (df_train2["exclude_x86"] != 1)] 

In [40]:
df_train3["file_hash"].nunique()

383

In [ ]:
#All the "exclude" labels need to be collapsed to "neither"
#df_train4["label"].value_counts()
#df_train3[df_train3["type"]=="Exclude"].index

In [41]:
df_train4 = df_train3.copy()
df_train4['broad_label'] = df_train4["type"].map(map_broad_label)

In [42]:
df_train4["broad_label"].value_counts()

broad_label
i_k         101
neither      75
receptor     63
i_other      57
i_na         51
i_ca         48
Name: count, dtype: int64

In [15]:
subtype_counts = pd.DataFrame(df_train4["label"].value_counts()).reset_index()
#list(subtype_counts[subtype_counts['count']>=10]['label'])
#subtype_counts[subtype_counts["count"] >= 10].sort_values(by="label").reset_index(drop=True)
#subtype_counts["new_label"] = subtype_counts["label"].map(map_new_label)
#subtype_counts["new_label"].unique()
#subtype_counts.groupby(["new_label","label"]).count()

In [43]:
df = df_train4.copy()

In [44]:
# Extract features from url
#df["heading"] = df["url"].apply(get_heading)  # Extract heading from webpage
#df["citation"] = df["heading"].apply(get_citation)
#df["first_author"] = df["citation"].apply(get_author)
#df["year"] = df["citation"].apply(get_year)  # Now handles multiple years

#df["dir"] = df["url"].apply(get_dir)
df["mod_name"] = df["url"].apply(get_fname)
#df["model_id"] = df["url"].apply(get_model_id)

#  Extract features from content
#df["title"] = df["content"].apply(get_title)
#df["comment"] = df["content"].apply(get_comment)
df["suffix"] = df["content"].apply(get_suffix)
#df["use_ion"] = df["content"].apply(get_use_ion)
df["read_ion"] = df["content"].apply(get_read_ion)
#Use empty list since cannot handle None
df['read_ion2'] = df['read_ion'].apply(lambda ion_list: list(set(map_ion(ion) for ion in ion_list)) if isinstance(ion_list, list) else [])
df["write_ion"] = df["content"].apply(get_write_ion)
df['write_ion2'] = df['write_ion'].apply(lambda ion_list: list(set(map_ion(ion) for ion in ion_list)) if isinstance(ion_list, list) else [])

#df["range"] = df["content"].apply(get_range)
#df["global"] = df["content"].apply(get_global)
df["parameter"] = df["content"].apply(get_parameter)
df["state"] = df["content"].apply(get_state)
df["net_receive"] = df["content"].apply(get_net_receive)
df["point_process"] = df["content"].apply(get_point_process)
df["nonspecific_current"] = df["content"].apply(get_nonspecific_current)

In [45]:
#Numeric Features
df["tau_min"], df["tau_max"] = zip(*df["parameter"].apply(get_tau))
df["e_min"], df["e_max"] = zip(*df["parameter"].apply(get_e))

#Discrete features
df["states_count"] = df["state"].apply(lambda x: len(x) if isinstance(x, list) else 0)

#Binary Features
df["clamp_yn"] = df.apply(lambda row: has_electrode_or_clamp(row["mod_name"], row["content"]), axis=1)
df["suffix_yn"] = df["suffix"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else (1 if pd.notna(x) else 0))
df["point_process_yn"] = df["point_process"].apply(lambda x: 1 if pd.notna(x) else 0)
df["net_receive_yn"] = df["net_receive"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else (1 if pd.notna(x) else 0))
df["i_nonspecific"] = df["nonspecific_current"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else (1 if pd.notna(x) else 0))

In [46]:
# Use MultiLabelBinarizer separately for read and write ions
mlb_read = MultiLabelBinarizer()
df_read_exploded = pd.DataFrame(mlb_read.fit_transform(df['read_ion2']), 
                                columns=[f'read_{col}_yn' for col in mlb_read.classes_])

mlb_write = MultiLabelBinarizer()
df_write_exploded = pd.DataFrame(mlb_write.fit_transform(df['write_ion2']), 
                                 columns=[f'write_{col}_yn' for col in mlb_write.classes_])

# Reset index before merging to avoid index mismatches
df = df.reset_index(drop=True)
df_read_exploded = df_read_exploded.reset_index(drop=True)
df_write_exploded = df_write_exploded.reset_index(drop=True)
df = df.drop(columns=['read_ion2', 'write_ion2']).join(df_read_exploded, rsuffix='_read_dup').join(df_write_exploded, rsuffix='_write_dup')

In [47]:
df.columns

Index(['row_id', 'file_hash', 'type', 'subtype_confidence', 'notes_free_text',
       'label', 'raw_sha', 'count', 'url', 'download_url', 'content',
       'error_code', 'exclude_error', 'exclude_x86', 'dataset', 'broad_label',
       'mod_name', 'suffix', 'read_ion', 'write_ion', 'parameter', 'state',
       'net_receive', 'point_process', 'nonspecific_current', 'tau_min',
       'tau_max', 'e_min', 'e_max', 'states_count', 'clamp_yn', 'suffix_yn',
       'point_process_yn', 'net_receive_yn', 'i_nonspecific', 'read_ca_i_yn',
       'read_ca_o_yn', 'read_cl_i_yn', 'read_cl_o_yn', 'read_e_ca_yn',
       'read_e_k_yn', 'read_e_na_yn', 'read_e_other_yn', 'read_i_cal_yn',
       'read_i_cl_yn', 'read_i_k_yn', 'read_i_na_yn', 'read_i_other_yn',
       'read_k_i_yn', 'read_k_o_yn', 'read_na_i_yn', 'read_na_o_yn',
       'read_other_i_yn', 'read_other_o_yn', 'write_ca_i_yn', 'write_cl_i_yn',
       'write_i_cal_yn', 'write_i_cl_yn', 'write_i_k_yn', 'write_i_na_yn',
       'write_i_other_yn', 

In [50]:
#keep features only
X_df = df.filter(regex=r'(_yn|_count|_min|_max|_id)$')

In [51]:
X_df

,row_id,tau_min,tau_max,e_min,e_max,states_count,clamp_yn,suffix_yn,point_process_yn,net_receive_yn,read_ca_i_yn,read_ca_o_yn,read_cl_i_yn,read_cl_o_yn,read_e_ca_yn,read_e_k_yn,read_e_na_yn,read_e_other_yn,read_i_cal_yn,read_i_cl_yn,read_i_k_yn,read_i_na_yn,read_i_other_yn,read_k_i_yn,read_k_o_yn,read_na_i_yn,read_na_o_yn,read_other_i_yn,read_other_o_yn,write_ca_i_yn,write_cl_i_yn,write_i_cal_yn,write_i_cl_yn,write_i_k_yn,write_i_na_yn,write_i_other_yn,write_k_o_yn,write_other_i_yn
0,1,NaN,NaN,45.00,45.00,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,NaN,NaN,-56.00,11.00,2,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,4,NaN,NaN,-80.00,-80.00,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,5,20.0,70.0,-75.00,-75.00,1,0,0,1,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,6,20.0,70.0,0.00,120.00,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,461,10.0,10.0,NaN,NaN,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
391,462,NaN,NaN,62.94,62.94,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
392,463,NaN,NaN,NaN,NaN,2,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
393,464,2.0,2.0,NaN,NaN,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [ ]:
my_filter = df["suffix"].apply(lambda x: x == None)

View(df.loc[my_filter,["url"]])

In [ ]:
View(df["first_atuhor"].value_counts())

In [ ]:
133/383

In [ ]:
# Flatten and collect unique ions from all three columns
unique_ions = set(df["write_ion"].dropna().explode()).union(
    set(df["read_ion"].dropna().explode()))

In [ ]:
df["mod_name"].unique()

In [ ]:
 set(df["suffix"].dropna().explode())

In [ ]:
View(df[df["read_ion2"].apply(lambda x: isinstance(x, list) and bool(set(x) & set(["unknown"])))]["read_ion"])


In [ ]:
View(df_train_raw[df_train_raw["label"]=="exclude_clamp"])                              

In [ ]:
df[df["has_electrode_or_clamp"]==1]

In [ ]:
# Apply function with unique category enforcement
df["read_ion"] = df["content"].apply(get_read_ion)
df["write_ion"] = df["content"].apply(get_write_ion)


In [ ]:
#View(df.loc[:,["read_ion","read_ion2"]])

In [ ]:
#METADATA
#model_id
#model_name
#author
#year

#ION-SPECIFIC FEATURES
#read_ion for major molecules (k, na, ca, cl, mg). if not major, then collapse into other
#write_ion for major molecules (k, na, ca, cl, mg), if not major, then collapse into other
#type of ion read for major molecules (outside concentration, inside concentration, reversal potential, current)
#type of ion written for major molecules (outside concentration, inside concentration, reversal potential, current)
#suffix: cad, nothing, kdr, na, kca
#min and max taus for each tau
#number of gates/states


#RECEPTOR-SPECIFIC FEATURES
#net_receive for major receptors. if not major then collapse into other
#point_process for major receptors. if not major then collapse into other

#NEITHER FEATURES
#has_electrode_current: has an ELECTRODE CURRENT in the mod file or a CLAMP in the mod file name 
#is_neither: does not read or write ions AND does not have a nonspecific current AND does not have a net receive 


In [ ]:
df


In [ ]:
View(df["suffix"].value_counts())

In [ ]:
View(df.loc[470,["url"]])

In [ ]:
#df[(df["mod_exclude"] == "not_found") | (df["mod_exclude"] == "x86_64")].shape

# Flagging Issues

In [ ]:
#row_id: 477 - commented out ranges included (https://modeldb.science/266818?tab=2&file=Ventricular_GUI.CircRes.ModelDB/Kss.mod)
#row_id: 481 - has comments with mod_state variables (https://modeldb.science/267511?tab=2&file=S1_Thal_NetPyNE_Frontiers_2022/sim/mod/ProbAMPANMDA_EMS.mod)
#row_id: 483 - has units in the mod_state ( https://modeldb.science/195666?tab=2&file=DewellGabbiani2018/mod_files/LGMD_KD_ca3.mod)
#row_id: 483 - was only extracting ONE parameter instead of multiple parameters (fixed)
#row_id 31 - has VALENCE in the write_ion (https://modeldb.science/116862?tab=2&file=b09jan13/IL3.mod)
#row_id 99 - need to fix use_ion

# Questions

In [ ]:
#todo: need to get INCLUDE for the notes table
#todo: should we collapse low-frequency labels
#todo: how do we process the SUFFIX, a lot of times the SUFFIX actually is the actual labeL?
#todo: do we include the name
#df["label"].value_counts()


In [ ]:
#REsol;ved Questions
#Is it okay that we make assumptions like start after READ and stop after WRITE, what if there is no WRITE statement?
#How to capture variables that are commented out vs. not? (don't capture stuff commented out)
#what's the best way to capture 


In [ ]:
View(df.loc[481,["url","mod_state"]],50)

In [ ]:
import re





In [ ]:
! git add .
! git commit -m "added features"
! git push 